<a href="https://colab.research.google.com/github/Snevj/SpecuativeWhisperUltra/blob/main/SpeculativeWhisper_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1 — Checking GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print(f' GPU ready: {torch.cuda.get_device_name(0)}')
else:
    print(' No GPU found — go to Runtime → Change runtime type → T4 GPU')


 GPU ready: Tesla T4


## Step 2 — Installing dependencies


In [ ]:
!pip install -q openai-whisper jiwer
!apt-get install -qq ffmpeg
print(' Dependencies installed')


 Dependencies installed


## Step 3 — Writing speculative_whisper.py

In [ ]:
%%writefile speculative_whisper.py
"""
speculative_whisper.py
======================
Speculative decoding with OpenAI Whisper.

Architecture
------------
* **Draft model** (Whisper Tiny)  – fast, cheap, generates a candidate token
  sequence autoregressively.
* **Verification model** (Whisper Large-V3) – runs a single *parallel* forward
  pass over the draft tokens and accepts / rejects each token via the classic
  speculative-decoding acceptance criterion (DeepMind / Google, 2023).

When a draft token is rejected the large model's own sample at that position is
used instead, so the output distribution exactly matches what large-only greedy
decoding would have produced.

References
----------
* Chen et al. (2023) "Accelerating Large Language Model Decoding with
  Speculative Sampling"  https://arxiv.org/abs/2302.01318
* Leviathan et al. (2023) "Fast Inference from Transformers via Speculative
  Decoding"  https://arxiv.org/abs/2211.17192
"""

from __future__ import annotations

import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional, Sequence, Tuple, Union

import numpy as np
import torch
import torch.nn.functional as F
import whisper
from whisper.audio import (
    N_FRAMES,
    HOP_LENGTH,
    SAMPLE_RATE,
    log_mel_spectrogram,
    pad_or_trim,
)
from whisper.decoding import DecodingOptions, DecodingResult
from whisper.tokenizer import get_tokenizer


# Public dataclass returned by transcribe()

@dataclass
class TranscriptionResult:
    audio_path: str
    text: str
    language: str
    latency_seconds: float
    draft_tokens_accepted: int = 0
    draft_tokens_generated: int = 0

    @property
    def acceptance_rate(self) -> float:
        if self.draft_tokens_generated == 0:
            return 0.0
        return self.draft_tokens_accepted / self.draft_tokens_generated


# Core class

class SpeculativeWhisper:
    """
    Speculative-decoding wrapper around the OpenAI Whisper models.

    Parameters
    ----------
    draft_model : str
        Whisper model name used as the draft / proposal model, e.g. ``"tiny"``.
    final_model : str
        Whisper model name used as the verification model, e.g. ``"large-v3"``.
    device : str
        ``"cuda"`` or ``"cpu"``.  Defaults to ``"cuda"`` when a GPU is available.
    speculation_length : int
        Number of draft tokens generated per speculative step (γ in the paper).
    language : str | None
        Force a particular language, or ``None`` to auto-detect.
    temperature : float
        Sampling temperature for the draft model.  Use 0 for greedy.
    verbose : bool
        Print per-file progress information.
    """

    def __init__(
        self,
        draft_model: str = "tiny",
        final_model: str = "large-v3",
        device: Optional[str] = None,
        speculation_length: int = 5,
        language: Optional[str] = None,
        temperature: float = 0.0,
        verbose: bool = True,
    ) -> None:
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.speculation_length = speculation_length
        self.language = language
        self.temperature = temperature
        self.verbose = verbose

        if self.verbose:
            print(f"[SpeculativeWhisper] Loading draft model  : whisper-{draft_model}")
        self.draft_model: whisper.Whisper = whisper.load_model(
            draft_model, device=device
        )
        self.draft_model.eval()

        if self.verbose:
            print(f"[SpeculativeWhisper] Loading verify model : whisper-{final_model}")
        self.final_model: whisper.Whisper = whisper.load_model(
            final_model, device=device
        )
        self.final_model.eval()

        # Tokenizer is shared (same vocabulary across all Whisper sizes)
        self.tokenizer = get_tokenizer(
            multilingual=self.final_model.is_multilingual,
            language=language,
            task="transcribe",
        )

    # Public API

    def transcribe(
        self,
        audio_files: Union[str, Sequence[str]],
        max_tokens: int = 448,
        batch_size: int = 1,
    ) -> List[TranscriptionResult]:
        """
        Transcribe one or more audio files using speculative decoding.

        Parameters
        ----------
        audio_files : str | list[str]
            Path(s) to .wav / .mp3 / .flac files accepted by whisper.load_audio.
        max_tokens : int
            Maximum number of new tokens to generate.
        batch_size : int
            How many files to process at once (currently each item in a batch
            is decoded independently; true batched speculative decoding is a
            future extension).

        Returns
        -------
        list[TranscriptionResult]
        """
        if isinstance(audio_files, (str, Path)):
            audio_files = [str(audio_files)]
        else:
            audio_files = [str(p) for p in audio_files]

        results: List[TranscriptionResult] = []

        for i in range(0, len(audio_files), batch_size):
            batch = audio_files[i : i + batch_size]
            for path in batch:
                result = self._transcribe_single(path, max_tokens=max_tokens)
                results.append(result)
                if self.verbose:
                    ar = f"{result.acceptance_rate:.1%}" if result.draft_tokens_generated else "n/a"
                    print(
                        f"  {Path(path).name} | {result.latency_seconds:.2f}s "
                        f"| acceptance={ar} | text={result.text[:60]!r}"
                    )

        return results

    # Baseline: standard large-model greedy decoding (for benchmarking)

    def transcribe_baseline(
        self,
        audio_files: Union[str, Sequence[str]],
    ) -> List[TranscriptionResult]:
        """
        Transcribe using standard Whisper Large greedy decoding (no speculation).
        Useful for speed / accuracy comparison.
        """
        if isinstance(audio_files, (str, Path)):
            audio_files = [str(audio_files)]
        else:
            audio_files = [str(p) for p in audio_files]

        results: List[TranscriptionResult] = []
        for path in audio_files:
            t0 = time.perf_counter()
            out = self.final_model.transcribe(path, language=self.language, temperature=0)
            latency = time.perf_counter() - t0
            results.append(
                TranscriptionResult(
                    audio_path=path,
                    text=out["text"].strip(),
                    language=out.get("language", ""),
                    latency_seconds=latency,
                )
            )
        return results

    # Internal helpers

    def _transcribe_single(self, path: str, max_tokens: int) -> TranscriptionResult:
        t0 = time.perf_counter()

        # ---- 1. Encode audio → mel spectrogram ----
        audio = whisper.load_audio(path)
        audio = pad_or_trim(audio)
        mel: torch.Tensor = log_mel_spectrogram(audio).to(self.device)
        mel = mel.unsqueeze(0)  # (1, n_mels, T)

        # ---- 2. Detect language (use large model for accuracy) ----
        if self.language:
            language = self.language
        else:
            language = self._detect_language(mel)

        # ---- 3. Build initial decoder token sequence ----
        sot_tokens = self.tokenizer.sot_sequence_including_notimestamps
        tokens: List[int] = list(sot_tokens)

        # ---- 4. Encode audio with *both* models once ----
        with torch.no_grad():
            draft_audio_feats = self.draft_model.encoder(
                mel.to(self.device)
            )
            final_audio_feats = self.final_model.encoder(
                mel.to(self.device)
            )

        # ---- 5. Speculative decoding loop ----
        total_accepted = 0
        total_draft = 0
        eot = self.tokenizer.eot

        while len(tokens) < len(sot_tokens) + max_tokens:
            # --- 5a. Draft γ tokens with the small model ---
            draft_tokens, draft_log_probs = self._draft_tokens(
                draft_audio_feats,
                tokens,
                gamma=self.speculation_length,
                eot=eot,
            )
            total_draft += len(draft_tokens)

            if not draft_tokens:
                break

            # --- 5b. Single parallel verification pass with large model ---
            verify_log_probs = self._verify_tokens(
                final_audio_feats,
                tokens,
                draft_tokens,
            )

            # --- 5c. Accept / reject via speculative sampling criterion ---
            accepted, bonus_token = self._accept_reject(
                draft_tokens,
                draft_log_probs,
                verify_log_probs,
            )
            total_accepted += len(accepted)
            tokens.extend(accepted)

            if bonus_token is not None:
                tokens.append(bonus_token)

            # Check for end-of-text
            if eot in accepted or bonus_token == eot:
                break

            # If we accepted fewer than γ we must also have the bonus token;
            # the loop naturally continues from there.

        latency = time.perf_counter() - t0

        # ---- 6. Decode tokens → text ----
        # Strip SOT prefix and EOT suffix
        text_tokens = [t for t in tokens[len(sot_tokens):] if t != eot]
        text = self.tokenizer.decode(text_tokens).strip()

        return TranscriptionResult(
            audio_path=path,
            text=text,
            language=language,
            latency_seconds=latency,
            draft_tokens_accepted=total_accepted,
            draft_tokens_generated=total_draft,
        )

    # ------------------------------------------------------------------

    def _detect_language(self, mel: torch.Tensor) -> str:
        """Use the large model to detect the spoken language."""
        with torch.no_grad():
            feats = self.final_model.encoder(mel)
            # Single forward pass of the first SOT token
            sot = torch.tensor([[self.tokenizer.sot]], device=self.device)
            logits = self.final_model.decoder(sot, feats)
            # Language tokens are at position 1 in the SOT sequence
            lang_tokens = self.final_model.decoder(
                torch.tensor([[self.tokenizer.sot]], device=self.device),
                feats,
            )
        # Fallback — whisper's built-in detect_language is more robust
        _, lang_probs = self.final_model.detect_language(mel)
        return max(lang_probs, key=lang_probs.get)

    # ------------------------------------------------------------------

    def _draft_tokens(
        self,
        audio_feats: torch.Tensor,
        context: List[int],
        gamma: int,
        eot: int,
    ) -> Tuple[List[int], List[torch.Tensor]]:
        """
        Autoregressively generate `gamma` tokens with the draft model.

        Returns
        -------
        draft_tokens   : list of int (may be shorter than γ if EOT encountered)
        draft_log_probs: list of 1-D probability tensors (vocab-size), one per token
        """
        draft_tokens: List[int] = []
        draft_log_probs: List[torch.Tensor] = []
        running_ctx = list(context)

        with torch.no_grad():
            for _ in range(gamma):
                ctx_tensor = torch.tensor(
                    [running_ctx], dtype=torch.long, device=self.device
                )
                logits = self.draft_model.decoder(ctx_tensor, audio_feats)
                logits = logits[0, -1, :]  # (vocab,)

                if self.temperature > 0:
                    probs = F.softmax(logits / self.temperature, dim=-1)
                    token = int(torch.multinomial(probs, 1).item())
                else:
                    token = int(logits.argmax().item())
                    probs = F.softmax(logits, dim=-1)

                draft_tokens.append(token)
                draft_log_probs.append(probs.cpu())
                running_ctx.append(token)

                if token == eot:
                    break

        return draft_tokens, draft_log_probs

    # ------------------------------------------------------------------

    def _verify_tokens(
        self,
        audio_feats: torch.Tensor,
        context: List[int],
        draft_tokens: List[int],
    ) -> List[torch.Tensor]:
        """
        Single parallel forward pass of the large (verification) model over
        the draft tokens.

        Returns a list of probability tensors (one per draft token position).
        """
        # The input is [ context | draft_tokens ] and we want the output
        # logits aligned with the draft token positions.
        input_ids = context + draft_tokens  # length = n_ctx + γ
        ctx_tensor = torch.tensor(
            [input_ids], dtype=torch.long, device=self.device
        )

        with torch.no_grad():
            logits = self.final_model.decoder(ctx_tensor, audio_feats)
            # logits shape: (1, seq_len, vocab)

        # We want the predictions at positions len(context)-1 … len(context)+γ-2
        # i.e., the distribution *after* each context token up to the last draft token
        offset = len(context) - 1  # position of the last context token's output
        verify_probs: List[torch.Tensor] = []
        for i in range(len(draft_tokens)):
            p = F.softmax(logits[0, offset + i, :], dim=-1).cpu()
            verify_probs.append(p)

        return verify_probs

    # ------------------------------------------------------------------

    @staticmethod
    def _accept_reject(
        draft_tokens: List[int],
        draft_probs: List[torch.Tensor],
        verify_probs: List[torch.Tensor],
    ) -> Tuple[List[int], Optional[int]]:
        """
        Classic speculative-decoding acceptance criterion.

        For each position i:
            r ~ Uniform(0, 1)
            if r < p_large(x_i) / p_draft(x_i)  → accept
            else → reject; sample bonus token from adjusted distribution

        Returns
        -------
        accepted     : tokens accepted from the draft
        bonus_token  : the bonus token sampled from the adjusted distribution
                       (None only if all γ tokens were accepted)
        """
        accepted: List[int] = []
        for i, token in enumerate(draft_tokens):
            p_draft = float(draft_probs[i][token].clamp(min=1e-9))
            p_large = float(verify_probs[i][token].clamp(min=0.0))

            accept_prob = min(1.0, p_large / p_draft)
            if torch.rand(1).item() < accept_prob:
                accepted.append(token)
            else:
                # Rejected: sample from adjusted distribution
                adjusted = (verify_probs[i] - draft_probs[i]).clamp(min=0.0)
                total = adjusted.sum()
                if total < 1e-9:
                    # Fallback: greedy from large model
                    bonus = int(verify_probs[i].argmax().item())
                else:
                    adjusted = adjusted / total
                    bonus = int(torch.multinomial(adjusted, 1).item())
                return accepted, bonus

        # All draft tokens accepted: sample one bonus token from large model
        bonus = int(verify_probs[-1].argmax().item())
        return accepted, bonus


Overwriting speculative_whisper.py


In [ ]:
# CELL: Define SpeculativeWhisper inline
import time
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple, Union
import torch
import torch.nn.functional as F
import whisper
from whisper.tokenizer import get_tokenizer
from whisper.audio import log_mel_spectrogram, pad_or_trim

@dataclass
class TranscriptionResult:
    audio_path: str
    text: str
    language: str
    latency_seconds: float
    draft_tokens_accepted: int = 0
    draft_tokens_generated: int = 0

    @property
    def acceptance_rate(self):
        if self.draft_tokens_generated == 0:
            return 0.0
        return self.draft_tokens_accepted / self.draft_tokens_generated


class SpeculativeWhisper:
    def __init__(self, draft_model="tiny", final_model="large-v3",
                 device=None, speculation_length=5,
                 language=None, temperature=0.0, verbose=True):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.speculation_length = speculation_length
        self.language = language
        self.temperature = temperature
        self.verbose = verbose
        if verbose: print(f"Loading draft  : whisper-{draft_model}")
        self.draft_model = whisper.load_model(draft_model, device=device).eval()
        if verbose: print(f"Loading verify : whisper-{final_model}")
        self.final_model = whisper.load_model(final_model, device=device).eval()
        self.tokenizer = get_tokenizer(
            multilingual=self.final_model.is_multilingual,
            language=language, task="transcribe")

    def _load_mel(self, path):
        audio = whisper.load_audio(path)
        audio = pad_or_trim(audio)
        mel = log_mel_spectrogram(audio, n_mels=128)  # 128 required for large-v3
        return mel.unsqueeze(0).to(self.device)

    def _detect_language(self, mel):
        _, lang_probs = self.final_model.detect_language(mel)
        return max(lang_probs, key=lang_probs.get)

    def _draft_tokens(self, audio_feats, context, gamma, eot):
        draft_tokens, draft_probs = [], []
        ctx = list(context)
        with torch.no_grad():
            for _ in range(gamma):
                t = torch.tensor([ctx], dtype=torch.long, device=self.device)
                logits = self.draft_model.decoder(t, audio_feats)[0, -1, :]
                probs = F.softmax(logits, dim=-1)
                token = int(logits.argmax().item())
                draft_tokens.append(token)
                draft_probs.append(probs.cpu())
                ctx.append(token)
                if token == eot:
                    break
        return draft_tokens, draft_probs

    def _verify_tokens(self, audio_feats, context, draft_tokens):
        ids = context + draft_tokens
        t = torch.tensor([ids], dtype=torch.long, device=self.device)
        with torch.no_grad():
            logits = self.final_model.decoder(t, audio_feats)
        offset = len(context) - 1
        return [F.softmax(logits[0, offset+i, :], dim=-1).cpu()
                for i in range(len(draft_tokens))]

    @staticmethod
    def _accept_reject(draft_tokens, draft_probs, verify_probs):
        accepted = []
        for i, token in enumerate(draft_tokens):
            p_d = float(draft_probs[i][token].clamp(min=1e-9))
            p_v = float(verify_probs[i][token].clamp(min=0.0))
            if torch.rand(1).item() < min(1.0, p_v / p_d):
                accepted.append(token)
            else:
                adj = (verify_probs[i] - draft_probs[i]).clamp(min=0.0)
                s = adj.sum()
                bonus = int((adj/s if s > 1e-9 else verify_probs[i]).argmax().item())
                return accepted, bonus
        return accepted, int(verify_probs[-1].argmax().item())

    def transcribe(self, audio_files, max_tokens=448, batch_size=1):
        if isinstance(audio_files, (str, Path)):
            audio_files = [str(audio_files)]
        return [self._transcribe_single(p, max_tokens) for p in audio_files]

    def transcribe_baseline(self, audio_files):
        if isinstance(audio_files, (str, Path)):
            audio_files = [str(audio_files)]
        results = []
        for path in audio_files:
            t0 = time.perf_counter()
            out = self.final_model.transcribe(path, language=self.language, temperature=0)
            results.append(TranscriptionResult(
                audio_path=path, text=out["text"].strip(),
                language=out.get("language", ""),
                latency_seconds=time.perf_counter()-t0))
        return results

    def _transcribe_single(self, path, max_tokens):
        t0 = time.perf_counter()
        mel = self._load_mel(path)
        language = self.language or self._detect_language(mel)
        sot = list(self.tokenizer.sot_sequence_including_notimestamps)
        tokens = list(sot)
        eot = self.tokenizer.eot

        with torch.no_grad():
            draft_feats = self.draft_model.encoder(mel)
            final_feats = self.final_model.encoder(mel)

        accepted_total = draft_total = 0
        while len(tokens) < len(sot) + max_tokens:
            dt, dp = self._draft_tokens(draft_feats, tokens, self.speculation_length, eot)
            draft_total += len(dt)
            if not dt: break
            vp = self._verify_tokens(final_feats, tokens, dt)
            acc, bonus = self._accept_reject(dt, dp, vp)
            accepted_total += len(acc)
            tokens.extend(acc)
            if bonus is not None: tokens.append(bonus)
            if eot in acc or bonus == eot: break

        text_tokens = [t for t in tokens[len(sot):] if t != eot]
        return TranscriptionResult(
            audio_path=path,
            text=self.tokenizer.decode(text_tokens).strip(),
            language=language,
            latency_seconds=time.perf_counter()-t0,
            draft_tokens_accepted=accepted_total,
            draft_tokens_generated=draft_total)

print("SpeculativeWhisper class defined")

SpeculativeWhisper class defined


In [ ]:
# ── Step 1: Free GPU memory ──────────────────────────────────
import torch, gc

# Delete old sw if it exists
try:
    del sw
except: pass

torch.cuda.empty_cache()
gc.collect()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

GPU free: 3.60 GB


In [ ]:
# ── Step 2: Load both models in float16 ──────────────────────
import whisper, torch

print("Loading whisper-tiny ...")
draft_model = whisper.load_model("tiny", device="cpu")   # load on CPU first
draft_model = draft_model.half().to("cuda")              # then move as float16

print("Loading whisper-large-v3 ...")
large_model = whisper.load_model("large-v3", device="cpu")
large_model = large_model.half().to("cuda")

print(f"GPU free after loading: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(" Both models loaded")

Loading whisper-tiny ...
Loading whisper-large-v3 ...
GPU free after loading: 3.60 GB
 Both models loaded


In [ ]:
# ── Step 3: Create sw using the already-loaded models ────────
from whisper.tokenizer import get_tokenizer

class SpeculativeWhisperLite(SpeculativeWhisper):
    """Same as SpeculativeWhisper but accepts pre-loaded model objects."""
    def __init__(self, draft_model, final_model, device="cuda",
                 speculation_length=5, language=None, temperature=0.0):
        # Skip __init__ model loading — use the ones we already loaded
        self.draft_model = draft_model
        self.final_model = final_model
        self.device = device
        self.speculation_length = speculation_length
        self.language = language
        self.temperature = temperature
        self.verbose = True
        self.tokenizer = get_tokenizer(
            multilingual=True, language=None, task="transcribe")

sw = SpeculativeWhisperLite(
    draft_model=draft_model,
    final_model=large_model,
    device="cuda",
    speculation_length=5,
)
print(" sw ready")

 sw ready


In [ ]:
sw = SpeculativeWhisper(
    draft_model='tiny',
    final_model='large-v3',
    device='cuda',
    speculation_length=5,
)

Loading draft  : whisper-tiny
Loading verify : whisper-large-v3


In [ ]:
def _detect_language_fixed(self, mel):
    # whisper.detect_language returns (token, [{lang: prob}, ...]) in some versions
    result = self.final_model.detect_language(mel)
    lang_probs = result[1]

    # Handle both dict and list-of-dicts formats
    if isinstance(lang_probs, list):
        lang_probs = lang_probs[0]   # unwrap the list

    return max(lang_probs, key=lang_probs.get)

# Patch the method on the existing sw instance
import types
sw._detect_language = types.MethodType(_detect_language_fixed, sw)
print(" Patched — now run the transcribe cell")

In [ ]:
AUDIO_FILE = 'audiofile.mp3'   # ← your filename here

results = sw.transcribe([AUDIO_FILE], max_tokens=200)
r = results[0]
print(f'Text      : {r.text}')
print(f'Latency   : {r.latency_seconds:.2f}s')
print(f'Acceptance: {r.acceptance_rate:.1%}')



In [ ]:
import types, torch

def _accept_reject_fixed(draft_tokens, draft_probs, verify_probs):
    accepted = []
    for i, token in enumerate(draft_tokens):
        # trim both to the same vocab size before comparing
        min_vocab = min(draft_probs[i].shape[0], verify_probs[i].shape[0])
        dp = draft_probs[i][:min_vocab]
        vp = verify_probs[i][:min_vocab]

        p_d = float(dp[token].clamp(min=1e-9)) if token < min_vocab else 1e-9
        p_v = float(vp[token].clamp(min=0.0))  if token < min_vocab else 0.0

        if torch.rand(1).item() < min(1.0, p_v / p_d):
            accepted.append(token)
        else:
            adj = (vp - dp).clamp(min=0.0)
            s = adj.sum()
            bonus = int((adj/s if s > 1e-9 else vp).argmax().item())
            return accepted, bonus

    return accepted, int(verify_probs[-1].argmax().item())

# patch as a regular method (it was @staticmethod before)
sw._accept_reject = _accept_reject_fixed
print("Patched, now running the transcribe cell")

In [ ]:
import types
from whisper.audio import log_mel_spectrogram, pad_or_trim
import whisper

def _transcribe_single_fixed(self, path, max_tokens):
    import time
    t0 = time.perf_counter()

    audio = whisper.load_audio(path)
    audio = pad_or_trim(audio)

    # ── each model gets its own mel with the correct n_mels ──
    mel_80  = log_mel_spectrogram(audio, n_mels=80).unsqueeze(0).to(self.device)
    mel_128 = log_mel_spectrogram(audio, n_mels=128).unsqueeze(0).to(self.device)

    # language detection uses large model → needs 128
    language = self.language or self._detect_language(mel_128)

    sot = list(self.tokenizer.sot_sequence_including_notimestamps)
    tokens = list(sot)
    eot = self.tokenizer.eot

    with torch.no_grad():
        draft_feats = self.draft_model.encoder(mel_80)   # tiny  → 80
        final_feats = self.final_model.encoder(mel_128)  # large → 128

    accepted_total = draft_total = 0
    while len(tokens) < len(sot) + max_tokens:
        dt, dp = self._draft_tokens(draft_feats, tokens, self.speculation_length, eot)
        draft_total += len(dt)
        if not dt: break
        vp = self._verify_tokens(final_feats, tokens, dt)
        acc, bonus = self._accept_reject(dt, dp, vp)
        accepted_total += len(acc)
        tokens.extend(acc)
        if bonus is not None: tokens.append(bonus)
        if eot in acc or bonus == eot: break

    from speculative_whisper import TranscriptionResult
    text_tokens = [t for t in tokens[len(sot):] if t != eot]
    return TranscriptionResult(
        audio_path=path,
        text=self.tokenizer.decode(text_tokens).strip(),
        language=language,
        latency_seconds=time.perf_counter()-t0,
        draft_tokens_accepted=accepted_total,
        draft_tokens_generated=draft_total)

sw._transcribe_single = types.MethodType(_transcribe_single_fixed, sw)
print("✅ Patched — now run the transcribe cell")

In [ ]:
!pip install -q gtts
from gtts import gTTS

sample_text = (
    "Speculative decoding dramatically accelerates large language model inference "
    "by using a smaller draft model to propose tokens that are then verified in "
    "parallel by the larger model."
)
tts = gTTS(sample_text, lang='en')
tts.save('audiofile.mp3')
print(' audiofile.mp3 created')

AUDIO_FILE = 'audiofile.mp3'
REFERENCE_TEXT = sample_text   # used for WER; set to None if unknown


## Step 5 — Load models

## Step 6 — Transcribe with speculative decoding

In [ ]:
results = sw.transcribe([AUDIO_FILE], max_tokens=200)
r = results[0]

print('─' * 60)
print(f'Text            : {r.text}')
print(f'Language        : {r.language}')
print(f'Latency         : {r.latency_seconds:.2f}s')
print(f'Draft tokens    : {r.draft_tokens_generated}')
print(f'Accepted        : {r.draft_tokens_accepted}  ({r.acceptance_rate:.1%})')


## Step 7 — Benchmark: speculative vs. baseline greedy

In [ ]:
import time

# ── Speculative ──
t0 = time.perf_counter()
spec_results = sw.transcribe([AUDIO_FILE], max_tokens=200)
spec_time = time.perf_counter() - t0

# ── Baseline (Large-V3 greedy) ──
t0 = time.perf_counter()
base_results = sw.transcribe_baseline([AUDIO_FILE])
base_time = time.perf_counter() - t0

speedup = base_time / max(spec_time, 1e-6)

print('─' * 60)
print(f'Speculative : {spec_time:.2f}s  →  "{spec_results[0].text[:80]}"..')
print(f'Baseline    : {base_time:.2f}s  →  "{base_results[0].text[:80]}"..')
print(f'Speedup     : {speedup:.2f}x')
print(f'Acceptance  : {spec_results[0].acceptance_rate:.1%}')


## Step 8 — WER comparison

In [ ]:
if REFERENCE_TEXT:
    import jiwer
    wer_spec = jiwer.wer(REFERENCE_TEXT, spec_results[0].text)
    wer_base = jiwer.wer(REFERENCE_TEXT, base_results[0].text)
    print(f'WER speculative : {wer_spec:.3f}')
    print(f'WER baseline    : {wer_base:.3f}')
    print('(Lower is better. Both should be nearly identical — speculative is lossless.)')
else:
    print('No reference text set — skipping WER.')


## Step 9 — Tune γ (speculation_length)


In [ ]:
print(f'{'γ':>4}  {'Latency':>10}  {'Acceptance':>12}  {'Speedup vs base':>16}')
print('─' * 50)

for gamma in [3, 5, 8, 12]:
    sw.speculation_length = gamma
    t0 = time.perf_counter()
    res = sw.transcribe([AUDIO_FILE], max_tokens=200)
    elapsed = time.perf_counter() - t0
    sp = base_time / max(elapsed, 1e-6)
    print(f'{gamma:>4}  {elapsed:>9.2f}s  {res[0].acceptance_rate:>11.1%}  {sp:>14.2f}x')
